# Inter-rater annotation agreement

Supply a CSV with one independently annotated scan per row:

`Subject, Scan, Dataset, RaterA, RaterB`

Mask paths can be absolute or relative to the CSV. Both masks must use the same label IDs and physical grid. Optional `AvailableA`/`AvailableB` columns contain JSON ID lists such as `[1, 3, 13, 21]`; otherwise all selected regions are assumed fully annotated. Empty annotated regions remain available. Do not substitute model predictions for a reader.

The module reports scan Dice/boundary errors and volume agreement. ICC(A,1) and its bootstrap interval count **animals**, averaging paired longitudinal volumes per animal. They require at least three animals; fewer animals remain explicitly marked. Reader A/B are symmetric for Dice and distances; sensitivity, precision and signed bias are directional.

Use the **rnet** kernel. Reporting is disabled until the manifest is configured.

In [ ]:
from pathlib import Path
import sys

REPO_ROOT: Path = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src/experiments/config.py").is_file()
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from IPython.display import display
import plotly.io as pio

pio.renderers.default = "vscode"
REFERENCE: Path = REPO_ROOT / "example/RARE/TBI_fm_19_49/Anat/TBI_fm_19_49_N4.nii.gz"


In [ ]:
from src.evaluation.inter_rater import load_annotation_pairs, inter_rater_report, inter_rater_slice_figure
from src.evaluation.statistics import AgreementConfig

PAIR_MANIFEST: Path | None = None
REGIONS: dict[int, str] = {
    1: "Lesion", 3: "Contra ventricle", 13: "Ipsi ventricle", 21: "Third ventricle",
}
SPATIAL_UNIT: str | None = None  # Use header units; override only when verified.
RUN_REPORT: bool = False
EXPORT_REPORT: bool = False
OUTPUT_DIR: Path = REPO_ROOT / "outputs/inter_rater/selected_annotations"
AGREEMENT = AgreementConfig(unit="mm3", bootstrap_samples=2000, confidence=0.95, seed=42)

# Complete these study details before manuscript reporting; files cannot establish them.
ANNOTATION_PROTOCOL: dict[str, str] = {
    "reader_a_experience": "",
    "reader_b_experience": "",
    "blinding_and_independence": "",
    "ambiguous_boundary_instructions": "",
    "disagreement_resolution": "",
}


In [ ]:
pairs = []
report = None
if RUN_REPORT:
    if PAIR_MANIFEST is None:
        raise ValueError("Set PAIR_MANIFEST to the paired annotation CSV")
    pairs = load_annotation_pairs(PAIR_MANIFEST)
    report = inter_rater_report(pairs, REGIONS, spatial_unit=SPATIAL_UNIT, agreement=AGREEMENT)
    report.settings["annotation_protocol"] = ANNOTATION_PROTOCOL
    display(report.metric_summary)
    display(report.volume_agreement)
    display(report.volume_pairs)
    display(report.scan_metrics)
else:
    print("Select paired reader annotations, then set RUN_REPORT=True.")


## Compare one pair

The masks are reoriented together to the same RARE reference used by the training notebooks (RIA; left-right axis 0). Display reorientation preserves physical coordinates and does not change the native-grid measurements. Choose another axis or slice and rerun the cell. No widget or HTML renderer is required.

In [ ]:
PAIR_INDEX: int = 0
VIEW_AXIS: int = 0
VIEW_POSITION: float = 0.5

if report is not None:
    figure = inter_rater_slice_figure(
        pairs[PAIR_INDEX], reference=REFERENCE, axis=VIEW_AXIS,
        position=VIEW_POSITION, spatial_unit=SPATIAL_UNIT,
    )
    figure.show(renderer="vscode")


## Export

Inspect both subject counts and per-region annotation availability before interpreting agreement. Surface distances remain undefined for empty masks. The volume ICC describes mean scan volumes per animal; it is not a longitudinal trajectory model. The exported source table records both reader paths and hashes. No automatic resampling, label swapping or adjudication is performed.

In [ ]:
if EXPORT_REPORT:
    if report is None:
        raise ValueError("Run the paired annotation report before exporting")
    saved_dir = report.export(OUTPUT_DIR)
    print("Saved to:", saved_dir)
